# Chapter 6: Real-life case study: road accidents in Al-Kharj, 2022

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/programming-for-data-science/blob/main/06_case_study_alkharj_accidents/case_study.ipynb)

This chapter is a complete data science project on **real data**: every road accident recorded in Al-Kharj in 2022, day by day, inside and outside the city. You will use everything from Chapters 2 to 5.

**The questions**

1. How many accidents, deaths and injuries happened in 2022, and where?
2. Are accidents outside the city more severe than inside it?
3. When do accidents happen: which months, which weekdays, and do Ramadan and public holidays change the picture?
4. Can we forecast tomorrow's number of accidents inside the city?

**Who would use the answers?** Traffic police planning patrols, the municipality prioritising road safety work, and hospitals planning emergency staffing.

**Project steps**

1. Load and understand the data
2. Audit data quality
3. Reshape and enrich with a calendar
4. Explore: totals, seasonality, weekdays, holidays
5. Severity: inside vs outside the city
6. Forecast daily accidents
7. Communicate the findings

> Cells marked **Exercise** are for you. In the student version they are empty; the instructor version contains a solution.

In [ ]:
# Data files live in the course repository. This works both in Colab and locally.
from pathlib import Path

BASE_URL = "https://raw.githubusercontent.com/alilouati/programming-for-data-science/main/data/"

def data_file(name):
    """Return a local path if the repo is cloned, otherwise the GitHub URL."""
    local = Path("../data") / name
    return local if local.exists() else BASE_URL + name

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

## 1. Load and understand the data

In [ ]:
raw = pd.read_csv(data_file("alkharj_accidents_2022.csv"))
raw.head()

In [ ]:
raw.info()

### Data dictionary

Each row is one day. Every measure exists twice: **OutsideCity** (roads outside the urban area, e.g. highways) and **WithinCity**.

| Column | Meaning |
|---|---|
| `Date` | day, written **day/month/year** |
| `CarAccidents…` | number of car accidents that day |
| `Deceased…` | number of **people killed** |
| `Fatalities…` | number of **accidents with at least one death** |
| `Injured…` | number of **people injured** |
| `AccidentsInjuries…` | number of **accidents with at least one injury** |

Source: Al-Kharj road accident records for 2022, provided by the course instructor.

### Parsing the date

`01/02/2022` could be 1 February or 2 January. The data dictionary says day first, so we state the format explicitly; never let pandas guess.

In [ ]:
df = raw.copy()
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df = df.set_index("Date").sort_index()
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "days")
print("one row per day:", df.index.is_unique and len(df) == 365)

Shorter column names make the code easier to read. We split each name into a **measure** and a **location**.

In [ ]:
measures = {"CarAccidents": "accidents", "Deceased": "deaths", "Fatalities": "fatal_accidents",
            "Injured": "injured", "AccidentsInjuries": "injury_accidents"}
places = {"OutsideCity": "outside", "WithinCity": "inside"}

new_names = {}
for col in df.columns:
    for m_old, m_new in measures.items():
        for p_old, p_new in places.items():
            if col == m_old + p_old:
                new_names[col] = f"{m_new}_{p_new}"
df = df.rename(columns=new_names)
df.describe().T.round(2)

**Exercise 1.** Without plotting anything yet: on an average day, how many accidents happen inside and outside the city? What is the busiest day of the year inside the city, and how many accidents happened that day?

In [ ]:
# TODO: your code here


## 2. Audit data quality

There are no missing values and every day appears once. But real data can be **wrong without being missing**. Each column has a meaning, so some combinations are impossible:

| Rule | Why |
|---|---|
| fatal accidents ≤ accidents | a fatal accident is an accident |
| injury accidents ≤ accidents | same |
| fatal accidents ≤ deaths | each fatal accident kills at least one person |
| deaths = 0 ⇒ fatal accidents = 0, and deaths > 0 ⇒ fatal accidents > 0 | same rule, other direction |
| injury accidents ≤ injured | each injury accident injures at least one person |
| injured > 0 ⇒ injury accidents > 0 | |

Let's encode these rules as boolean columns and count the violations.

In [ ]:
def audit(data, place):
    a = data[f"accidents_{place}"]
    d = data[f"deaths_{place}"]
    fa = data[f"fatal_accidents_{place}"]
    i = data[f"injured_{place}"]
    ia = data[f"injury_accidents_{place}"]
    checks = pd.DataFrame({
        "fatal > accidents": fa > a,
        "injury acc. > accidents": ia > a,
        "fatal acc. > deaths": fa > d,
        "deaths but no fatal acc.": (d > 0) & (fa == 0),
        "injury acc. > injured": ia > i,
        "injured but no injury acc.": (i > 0) & (ia == 0),
    })
    checks.columns = pd.MultiIndex.from_product([[place], checks.columns])
    return checks

checks = pd.concat([audit(df, "outside"), audit(df, "inside")], axis=1)
checks.sum().to_frame("violations")

In [ ]:
bad_days = checks.any(axis=1)
print(bad_days.sum(), "days break at least one rule")
df.loc[bad_days].filter(like="outside").join(df.loc[bad_days].filter(like="inside"))

Look at **27 January**: outside the city, 3 deaths but 0 fatal accidents; inside the city, 1 fatal accident but 0 deaths. A likely explanation is that one accident was recorded under the wrong location in one of the columns. Several days also report injured people with no injury accident, or injury accidents on days with no accidents at all.

**What should we do?** There is no single right answer:

- **Keep** the rows (they are a small share of the data) and remember the counts are approximate.
- **Correct** them, only with evidence, e.g. by asking the data owner.
- **Exclude** them from analyses where they matter (severity rates), and keep them elsewhere.

We will flag them, keep them, and report results with and without them where it matters. Writing down this decision is part of the analysis.

In [ ]:
df["quality_flag"] = bad_days
df["quality_flag"].mean().round(3)

**Exercise 2.** The rules above compare columns within a location. Can you think of a rule that compares **inside with outside**, or a day with the days around it? Implement one and count how many days it flags.

In [ ]:
# TODO: your code here


## 3. Reshape and enrich

### Wide to long

The table is **wide**: one column per (measure, location). For grouping and plotting by location, a **long** table with a `location` column is easier.

In [ ]:
long = (df.drop(columns="quality_flag")
          .reset_index()
          .melt(id_vars="Date", var_name="variable", value_name="count"))
long[["measure", "location"]] = long["variable"].str.rsplit("_", n=1, expand=True)
long = long.drop(columns="variable")
long.head()

In [ ]:
long.pivot_table(index="measure", columns="location", values="count", aggfunc="sum")

### Calendar features

Traffic depends on the calendar. We add the weekday, the month and the Saudi weekend (Friday and Saturday).

In [ ]:
df["weekday"] = df.index.day_name()
df["month"] = df.index.month
df["month_name"] = df.index.month_name().str[:3]
df["weekend"] = df.index.day_of_week.isin([4, 5])   # Monday = 0, so Friday = 4, Saturday = 5
df[["weekday", "month", "weekend"]].head(8)

### Ramadan and public holidays (merging a second table)

Holidays are not in the data, so we build a small **events table** and merge it. Dates for 2022 in Saudi Arabia:

- **Founding Day**: 22 February
- **Ramadan 1443**: 2 April to 1 May
- **Eid al-Fitr**: from 2 May (we flag the first three days)
- **Eid al-Adha**: from 9 July (first three days)
- **National Day**: 23 September
- **World Cup holiday**: 23 November, declared the day after Saudi Arabia beat Argentina

> Check these dates against an official calendar before using them for a real decision; Hijri dates depend on moon sighting.

In [ ]:
events = pd.DataFrame([
    ("Founding Day",      "2022-02-22", "2022-02-22"),
    ("Ramadan",           "2022-04-02", "2022-05-01"),
    ("Eid al-Fitr",       "2022-05-02", "2022-05-04"),
    ("Eid al-Adha",       "2022-07-09", "2022-07-11"),
    ("National Day",      "2022-09-23", "2022-09-23"),
    ("World Cup holiday", "2022-11-23", "2022-11-23"),
], columns=["event", "start", "end"])

# One row per day of each event
event_days = pd.concat(
    [pd.DataFrame({"Date": pd.date_range(row.start, row.end), "event": row.event})
     for row in events.itertuples()],
    ignore_index=True)

df = (df.reset_index()
        .merge(event_days, on="Date", how="left", validate="one_to_one")
        .set_index("Date"))
df["event"] = df["event"].fillna("Normal day")
df["ramadan"] = df["event"] == "Ramadan"
df["holiday"] = df["event"].isin(["Founding Day", "Eid al-Fitr", "Eid al-Adha", "National Day", "World Cup holiday"])
df["event"].value_counts()

Totals across locations are useful too:

In [ ]:
for m in ["accidents", "deaths", "injured"]:
    df[f"{m}_total"] = df[f"{m}_inside"] + df[f"{m}_outside"]

## 4. Explore

### 4.1 The year in numbers

In [ ]:
totals = df[["accidents_inside", "accidents_outside", "deaths_inside", "deaths_outside",
             "injured_inside", "injured_outside"]].sum()
summary = pd.DataFrame({
    "inside": totals.filter(like="inside").values,
    "outside": totals.filter(like="outside").values,
}, index=["accidents", "deaths", "injured"])
summary["total"] = summary.sum(axis=1)
summary["share_outside_%"] = (100 * summary["outside"] / summary["total"]).round(1)
summary

About **15%** of accidents happen outside the city, but they account for more than **half of the deaths**. We come back to this in section 5.

### 4.2 Over the year

Daily counts are noisy. A **7-day rolling mean** shows the trend and also removes the weekly cycle.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, place in zip(axes, ["inside", "outside"]):
    s = df[f"accidents_{place}"]
    ax.plot(s.index, s, color="0.75", linewidth=0.8, label="daily")
    ax.plot(s.index, s.rolling(7, center=True).mean(), linewidth=2, label="7-day mean")
    ax.axvspan(pd.Timestamp("2022-04-02"), pd.Timestamp("2022-05-01"), color="tab:green", alpha=0.12, label="Ramadan")
    ax.set_ylabel("Accidents per day")
    ax.set_title(f"Accidents {place} the city, 2022")
axes[0].legend(loc="upper left", ncols=3, frameon=False)
plt.tight_layout()
plt.show()

### 4.3 By month

Months have different numbers of days, so compare the **daily average**, not the total.

In [ ]:
monthly = df.groupby(["month", "month_name"])[["accidents_inside", "accidents_outside"]].mean().round(2)
monthly = monthly.droplevel("month")
ax = monthly.plot.bar(figsize=(10, 4), rot=0)
ax.set_ylabel("Mean accidents per day")
ax.set_xlabel("")
ax.set_title("Average daily accidents by month")
ax.legend(["inside the city", "outside the city"], ncols=2, frameon=False,
          loc="upper center", bbox_to_anchor=(0.5, -0.08))
plt.show()
monthly.T

**July stands out**: about 10 accidents per day inside the city, a third below the rest of the year. A likely explanation is Eid al-Adha (9 July) and the summer holidays, when many residents travel away from Al-Kharj. January is the busiest month.

**Exercise 3.** Which month has the highest daily average inside the city? Is it the same month as the highest **total**? Why might the two disagree?

In [ ]:
# TODO: your code here


### 4.4 By weekday

In [ ]:
order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=df.reset_index(), x="weekday", y="accidents_inside", order=order, errorbar=("ci", 95), ax=ax)
ax.set_ylabel("Accidents inside the city per day")
ax.set_xlabel("")
ax.set_title("Accidents by weekday (bars: mean, lines: 95% confidence interval)")
plt.show()

Differences between weekdays are small, and the confidence intervals overlap. Is the **weekend** different from working days?

In [ ]:
weekend = df.loc[df["weekend"], "accidents_inside"]
workday = df.loc[~df["weekend"], "accidents_inside"]
t = stats.ttest_ind(weekend, workday, equal_var=False)
print(f"weekend mean {weekend.mean():.2f} vs workday mean {workday.mean():.2f}, p = {t.pvalue:.3f}")

### 4.5 A calendar heatmap

A heatmap with weeks as columns and weekdays as rows shows the whole year at once.

In [ ]:
cal = df[["accidents_inside"]].copy()
cal["week"] = cal.index.isocalendar().week.astype(int)
cal.loc[(cal.index.month == 1) & (cal["week"] > 50), "week"] = 0     # first days of January
cal["weekday"] = pd.Categorical(cal.index.day_name(), categories=order, ordered=True)
grid = cal.pivot_table(index="weekday", columns="week", values="accidents_inside", observed=False)

fig, ax = plt.subplots(figsize=(14, 3.2))
sns.heatmap(grid, cmap="Blues", cbar_kws={"label": "accidents"}, ax=ax, xticklabels=4)
ax.set_title("Accidents inside the city, 2022 (each cell is one day)")
ax.set_xlabel("ISO week")
ax.set_ylabel("")
plt.show()

### 4.6 Ramadan and holidays

In [ ]:
by_event = (df.groupby("event")[["accidents_inside", "accidents_outside", "deaths_total"]]
              .agg(["mean", "count"]).round(2))
by_event

In [ ]:
ram = df.loc[df["ramadan"], "accidents_inside"]
rest = df.loc[~df["ramadan"] & ~df["holiday"], "accidents_inside"]
t = stats.ttest_ind(ram, rest, equal_var=False)
print(f"Ramadan {ram.mean():.2f} vs normal days {rest.mean():.2f} accidents/day inside the city, p = {t.pvalue:.3f}")

Accidents inside the city were **higher during Ramadan** (about 17 per day vs 15.5), and the difference is unlikely to be chance (p < 0.05). Eid days are quieter. Holidays are few days, so their averages are uncertain.

**Exercise 4.** During Ramadan, traffic peaks shift towards iftar and late evening. Our data is daily, so we cannot see hours. Write down which **extra data** would let you test whether the Ramadan increase is concentrated just before iftar, and how you would join it to this table.

*Your answer here.*

## 5. Severity: inside vs outside the city

Counting accidents is not enough: an accident on a highway at 120 km/h is very different from a parking collision. We compare **rates**:

- deaths per 100 accidents
- injured per 100 accidents
- share of accidents that are fatal

In [ ]:
def severity(data):
    rows = {}
    for place in ["inside", "outside"]:
        a = data[f"accidents_{place}"].sum()
        rows[place] = {
            "accidents": a,
            "deaths per 100 accidents": 100 * data[f"deaths_{place}"].sum() / a,
            "injured per 100 accidents": 100 * data[f"injured_{place}"].sum() / a,
            "% of accidents fatal": 100 * data[f"fatal_accidents_{place}"].sum() / a,
        }
    return pd.DataFrame(rows).T.round(2)

sev_all = severity(df)
sev_clean = severity(df[~df["quality_flag"]])
pd.concat({"all days": sev_all, "flagged days removed": sev_clean})

Whichever way we treat the flagged days, an accident **outside** the city is about **8 times more likely to kill** someone than one inside. Is that difference statistically significant? A **chi-square test** compares the proportion of fatal accidents in the two locations.

In [ ]:
table = pd.DataFrame({
    "fatal": [df["fatal_accidents_inside"].sum(), df["fatal_accidents_outside"].sum()],
    "not fatal": [df["accidents_inside"].sum() - df["fatal_accidents_inside"].sum(),
                  df["accidents_outside"].sum() - df["fatal_accidents_outside"].sum()],
}, index=["inside", "outside"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(table)
print(f"chi2 = {chi2:.1f}, p = {p:.1e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
rate = sev_all["deaths per 100 accidents"]
ax.barh(rate.index, rate.values, color=["tab:blue", "tab:red"])
for y, v in enumerate(rate.values):
    ax.text(v + 0.05, y, f"{v:.2f}", va="center")
ax.set_xlabel("Deaths per 100 accidents")
ax.set_title("Accidents outside the city are far deadlier")
ax.grid(axis="y", visible=False)
plt.show()

**Exercise 5.** Deaths are rare events (62 in the whole year), so monthly death counts are very noisy. Compute the monthly number of deaths outside the city and plot it with a bar chart. Would you trust a headline such as "deaths outside the city tripled from March to April"? Why not?

In [ ]:
# TODO: your code here


## 6. Forecast daily accidents inside the city

Could the police plan tomorrow's patrols from a forecast? We try to predict `accidents_inside` from calendar features only.

**Rules for an honest forecast**

- Train on the past, test on the future: **January to September** for training, **October to December** for testing. Never shuffle time series.
- Compare with a **baseline**: a model that always predicts the training mean. A model is only useful if it beats the baseline.

In [ ]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error

features = pd.get_dummies(df[["weekday", "month_name"]], drop_first=True, dtype=float)
features["ramadan"] = df["ramadan"].astype(float)
features["holiday"] = df["holiday"].astype(float)
features["trend"] = np.arange(len(df)) / len(df)      # 0 on 1 January, ~1 on 31 December
target = df["accidents_inside"]

train = df.index < "2022-10-01"
X_train, X_test = features[train], features[~train]
y_train, y_test = target[train], target[~train]
print(X_train.shape, X_test.shape)

Accident counts are whole numbers, never negative, and roughly follow a **Poisson** distribution, so we use a Poisson regression. Month dummies cannot be learnt for October to December (those months are not in the training data), so we leave month out.

We fit two models:

- **Calendar model**: weekday, Ramadan, holiday
- **Calendar + trend**: the same plus a linear trend over the year

In [ ]:
calendar_cols = [c for c in features.columns if c.startswith("weekday")] + ["ramadan", "holiday"]
trend_cols = calendar_cols + ["trend"]

model = PoissonRegressor(alpha=1e-3, max_iter=1000).fit(X_train[calendar_cols], y_train)
model_trend = PoissonRegressor(alpha=1e-3, max_iter=1000).fit(X_train[trend_cols], y_train)

pred = pd.Series(model.predict(X_test[calendar_cols]), index=y_test.index)
pred_trend = pd.Series(model_trend.predict(X_test[trend_cols]), index=y_test.index)
baseline = pd.Series(y_train.mean(), index=y_test.index)
last_week = target.shift(7)[~train]                      # "same day last week"

results = pd.Series({
    "baseline: training mean": mean_absolute_error(y_test, baseline),
    "baseline: same day last week": mean_absolute_error(y_test, last_week),
    "Poisson: calendar": mean_absolute_error(y_test, pred),
    "Poisson: calendar + trend": mean_absolute_error(y_test, pred_trend),
}).round(2)
results.to_frame("mean absolute error (accidents/day)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test, color="0.6", label="actual")
ax.plot(pred.index, pred, linewidth=2, label="Poisson: calendar")
ax.plot(pred_trend.index, pred_trend, linewidth=2, linestyle=":", label="Poisson: calendar + trend")
ax.axhline(y_train.mean(), color="tab:orange", linestyle="--", label="training mean")
ax.set_ylabel("Accidents inside the city")
ax.set_title("Forecast vs actual, October to December 2022")
ax.legend(frameon=False, ncols=4)
plt.show()

In [ ]:
coef = pd.Series(np.exp(model.coef_), index=calendar_cols).sort_values()
coef.round(3).to_frame("multiplicative effect on expected accidents")

**What did we learn?**

- The calendar model is **no better than predicting the training average every day**. The coefficients point the right way (holidays lower, Ramadan higher), but the effects are small compared with the day-to-day noise.
- Adding a trend makes the forecast **worse**: the model learnt a slight decline from January to September and extended it into a period where accidents actually rose. Extrapolating a trend is risky.
- "Same day last week" is the worst: it copies last week's noise.
- Daily counts have a lot of randomness. Their variance is close to their mean, which is the signature of Poisson noise: even a perfect model would be off by about ±3 accidents on a typical day.
- To forecast better we need **new information**, not a fancier model: weather, traffic volume, school calendar, events, road works.

That is a useful, honest result. Reporting "we cannot forecast daily counts much better than the average with calendar data" is better than overselling a model.

**Exercise 6.** Try to beat the training-mean baseline. Ideas: stronger regularisation (`alpha` = 0.1, 1, 10), the training **median** instead of the mean, or the mean of the last 28 days. Report the mean absolute error of each idea in one table. Which would you deploy, and why?

In [ ]:
# TODO: your code here


## 7. Communicate the findings

A data science project ends with a clear message for the people who will act on it.

**Al-Kharj road accidents 2022: key findings**

1. **6,719 accidents** were recorded, **62 people died** and **307 were injured**.
2. **Outside the city is where people die.** Only 15% of accidents happen outside the city, but they cause more than half of the deaths: about 3.6 deaths per 100 accidents vs 0.45 inside, roughly 8 times higher (chi-square p < 0.001).
3. **Ramadan is busier.** Inside the city, accidents rose from about 15.7 to 17.1 per day during Ramadan (p ≈ 0.04). Eid holidays were quieter.
4. **July is the quietest month** (about 10 accidents per day inside the city, vs 15-18 in other months), probably because of Eid al-Adha and summer travel.
5. **Weekdays barely matter.** Daily averages range only between 15 and 16.5 accidents, and weekends are not significantly different.
6. **Daily counts cannot be forecast** better than the yearly average from the calendar alone; staffing should plan for the average plus a buffer of about 6 accidents (the 95th percentile is about 22 per day inside the city).
7. **Data quality:** 20 days contain impossible combinations (e.g. deaths without a fatal accident). The findings above hold with or without those days, but the recording process should be checked.

**Recommendations**

- Focus road-safety resources (speed enforcement, lighting, emergency response time) on **roads outside the city**.
- Strengthen patrols and awareness campaigns **during Ramadan**.
- Add validation rules at data entry (section 2) so impossible records are caught when they are typed.
- Collect hourly data, accident location and weather to support better analysis next year.

In [ ]:
check = {
    "accidents": int(df["accidents_total"].sum()),
    "deaths": int(df["deaths_total"].sum()),
    "injured": int(df["injured_total"].sum()),
    "flagged days": int(df["quality_flag"].sum()),
    "95th percentile inside": df["accidents_inside"].quantile(0.95),
}
check

## Mini-project (groups of 2-3)

Choose one and present your results in 10 minutes:

1. **Weather.** Download daily temperature, wind and dust data for Al-Kharj in 2022 (e.g. from the [Open-Meteo historical API](https://open-meteo.com/en/docs/historical-weather-api)), merge it on the date and test whether hot or dusty days have more accidents.
2. **Severity model.** Model the daily number of deaths with a Poisson regression. Which features matter?
3. **Another year.** If the 2023 data becomes available, compare it with 2022. Did the outside-the-city death rate change?
4. **Dashboard.** Build an interactive Plotly chart where the user picks the measure (accidents, deaths, injured) and location.